<a href="https://colab.research.google.com/github/Seavhab-heng/aispeak/blob/main/Another_copy_of_HSllm_Google_Colab_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 👑 HSllm AI Model Training Pipeline (Google Colab GPU)
Fine-Tuning custom Universal & Khmer AI Model **HSllm** using **mistralai/Mistral-7B-Instruct-v0.3** and LoRA / QLoRA.

In [ ]:
# 1. Install High-Performance AI Training Dependencies
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "xformers" "trl<0.16.0" peft accelerate bitsandbytes datasets

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 109.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 99.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 86.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 21.0 MB/s eta 0:

In [ ]:
# 2. Load Base Foundation Model with 4-bit Quantization
import torch
from unsloth import FastLanguageModel

max_seq_length = 2048
model_name = 'mistralai/Mistral-7B-Instruct-v0.3'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=None, # Auto detection (Float16 or Bfloat16)
    load_in_4bit=True,
)

# 3. Add LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing=True,
    random_state=3407,
)
print('✓ Model & LoRA initialized successfully!')

==((====))==  Unsloth 2026.8.22: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

✓ Model & LoRA initialized successfully!


In [ ]:
# 4. Load Training Dataset (chat_template_train.jsonl)
from datasets import load_dataset

# If running directly in Colab, ensure chat_template_train.jsonl is uploaded
dataset = load_dataset('json', data_files='chat_template_train.jsonl', split='train')
print(f'✓ Loaded {len(dataset):,} training conversation samples!')

Generating train split: 0 examples [00:00, ? examples/s]

✓ Loaded 9,195 training conversation samples!


In [ ]:
# 5. Format Dataset using Tokenizer Chat Template
def format_prompts(examples):
    texts = []
    for msgs in examples['messages']:
        text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return {'text': texts}

dataset = dataset.map(format_prompts, batched=True)

Map:   0%|          | 0/9195 [00:00<?, ? examples/s]

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=max_seq_length,
    tokenizer=tokenizer, # Explicitly pass the tokenizer
    args=SFTConfig(
        output_dir='HSllm_outputs',
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        seed=3407,
    ),
)

print('🚀 Starting Training...')
trainer.train()
print('🎉 Training Complete!')

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/9195 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
🚀 Starting Training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 9,195 | Num Epochs = 3 | Total steps = 3,450
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 7,289,966,592 (0.58% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,1.412708
2,1.421929
3,1.296933
4,1.178550
5,0.971627
6,0.851233
7,0.665778
8,0.512611
9,0.386697
10,0.279046


Unsloth: Restored added_tokens_decoder metadata in HSllm_outputs/checkpoint-500/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in HSllm_outputs/checkpoint-500.


Step,Training Loss
1,1.412708
2,1.421929
3,1.296933
4,1.178550
5,0.971627
6,0.851233
7,0.665778
8,0.512611
9,0.386697
10,0.279046


KeyboardInterrupt: 

In [ ]:
# 7. Save LoRA Weights and Export to 16-bit / GGUF (for Ollama)
model.save_pretrained_merged('HSllm_lora_model', tokenizer, save_method='lora')
# Optional: Export to GGUF format for Ollama / llama.cpp
# model.save_pretrained_gguf('HSllm_GGUF', tokenizer, quantization_method='q4_k_m')
print('✓ Model saved successfully! Ready to download and use in Ollama.')

config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in HSllm_lora_model/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in HSllm_lora_model.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

...



Unsloth: Preparing safetensor model files:  67%|██████▋   | 2/3 [04:04<02:00, 120.90s/it]

model-00003-of-00003.safetensors: reconstructing file:   0%|          |  0.00B / 4.55GB            

model-00003-of-00003.safetensors: downloading bytes:           |  0.00B            



Unsloth: Merging weights into 16bit: 100%|██████████| 3/3 [05:02<00:00, 100.70s/it]


Unsloth: Merge process complete. Saved to `/content/HSllm_lora_model`
✓ Model saved successfully! Ready to download and use in Ollama.


In [ ]:
# Zip and Download your trained HSllm model to your PC
import shutil
from google.colab import files
import os

print('Listing contents of /content/ for debugging:')
!ls -l /content/

# Ensure the directory exists before attempting to archive
# If it still fails, the problem is in the model saving step (cell XZlqANVzWC9L).
if not os.path.exists('/content/HSllm_lora_model'):
    print('Error: The model directory /content/HSllm_lora_model was not found. Please ensure the model saving step (cell XZlqANVzWC9L) completed successfully.')
else:
    shutil.make_archive('HSllm_final_model', 'zip', root_dir='/content', base_dir='HSllm_lora_model')
    files.download('HSllm_final_model.zip')
    print('🎉 Downloading your trained HSllm model...')


Listing contents of /content/ for debugging:
total 4
drwxr-xr-x 1 root root 4096 Aug 24 13:21 sample_data
Error: The model directory /content/HSllm_lora_model was not found. Please ensure the model saving step (cell XZlqANVzWC9L) completed successfully.
